In [300]:
import numpy as np

In [301]:
# Problem 0:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 3
NUM_ITEMS = 10
NUM_DIMENSIONS = 2
VALUES = rng.integers(0, 100, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 100, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    0, 100 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

In [302]:
# Problem 1:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS1 = 3
NUM_ITEMS1 = 20
NUM_DIMENSIONS1 = 2
VALUES1 = rng.integers(0, 100, size=NUM_ITEMS1)
WEIGHTS1 = rng.integers(0, 100, size=(NUM_ITEMS1, NUM_DIMENSIONS1))
CONSTRAINTS1 = rng.integers(
    0, 100 * NUM_ITEMS1 // NUM_KNAPSACKS1, size=(NUM_KNAPSACKS1, NUM_DIMENSIONS1)
)

In [303]:
# Problem 2:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS2 = 10
NUM_ITEMS2 = 100
NUM_DIMENSIONS2 = 10
VALUES2 = rng.integers(0, 1000, size=NUM_ITEMS2)
WEIGHTS2 = rng.integers(0, 1000, size=(NUM_ITEMS2, NUM_DIMENSIONS2))
CONSTRAINTS2 = rng.integers(
    1000 * 2, 1000 * NUM_ITEMS2 // NUM_KNAPSACKS2, size=(NUM_KNAPSACKS2, NUM_DIMENSIONS2)
)

In [304]:
# Problem 3:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS3 = 100
NUM_ITEMS3 = 5000
NUM_DIMENSIONS3 = 100
VALUES3 = rng.integers(0, 1000, size=NUM_ITEMS3)
WEIGHTS3 = rng.integers(0, 1000, size=(NUM_ITEMS3, NUM_DIMENSIONS3))
CONSTRAINTS3 = rng.integers(
    1000 * 10, 1000 * 2 * NUM_ITEMS3 // NUM_KNAPSACKS3, size=(NUM_KNAPSACKS3, NUM_DIMENSIONS3)
)

In [305]:
def is_valid(solution, num_knapsacks, weights, constraints):
    if not np.all(solution.sum(axis=0) <= 1):
        return False

    for k in range(num_knapsacks):
        items_in_knapsack_k = solution[k]
        weight_of_knapsack_k = weights[items_in_knapsack_k].sum(axis=0)
        if not np.all(weight_of_knapsack_k <= constraints):
            return False

    return True

def validate(solution):
    if not np.all(solution.sum(axis=0) <= 1):
        return False #if an item is in more than one knapsack exit
    
    for nap in range(NUM_KNAPSACKS):
        weights = WEIGHTS[solution[nap]].sum(axis=0)
        if np.any(weights > CONSTRAINTS[nap]):
            return False #if invalid exit
    return True

In [306]:
def evaluate(solution, values, num_knapsacks, weights, constraints):
    if not is_valid(solution, num_knapsacks, weights, constraints):
        return -1.0
    items_placed = np.any(solution, axis=0)
    total_value = values[items_placed].sum()
    return float(total_value)

def cost(solution):
    all_knapsacks = np.any(solution, axis=0)
    all_cost = VALUES[all_knapsacks].sum()
    return all_cost

In [307]:
def move(solution, num_items, num_knapsacks):
    neighbor = solution.copy()
    item_to_move = rng.integers(0, num_items)
    new_knapsack_idx = rng.integers(-1, num_knapsacks)
    neighbor[:, item_to_move] = False
    if new_knapsack_idx != -1:
        neighbor[new_knapsack_idx, item_to_move] = True

    return neighbor

def tweak(solution, p = 0.4):
    new_solution = solution.copy()
    if np.random.random() < p:
        # Swap two items between two knapsacks
        knapsack1, knapsack2 = np.random.choice(NUM_KNAPSACKS, size=2, replace=False)
        item1 = np.random.randint(0, NUM_ITEMS)
        item2 = np.random.randint(0, NUM_ITEMS)
        new_solution[knapsack1, item1], new_solution[knapsack2, item2] = (
            new_solution[knapsack2, item2],
            new_solution[knapsack1, item1],
        )
    else:
        # Add or remove an item from a knapsack
        knapsack = np.random.randint(0, NUM_KNAPSACKS)
        item = np.random.randint(0, NUM_ITEMS)
        new_solution[knapsack, item] = not new_solution[knapsack, item]
    
    return new_solution

In [308]:
def hill_climber_fast(initial_solution: np.ndarray, max_steps_no_improvement: int, values, num_knapsacks, weights,
                      constraints, num_items):
    current_solution, current_score = initial_solution, evaluate(initial_solution, values, num_knapsacks, weights,
                                                                 constraints)
    steps_without_improvement = 0
    while steps_without_improvement < max_steps_no_improvement:
        neighbor = move(current_solution, num_items, num_knapsacks)
        neighbor_score = evaluate(neighbor, values, num_knapsacks, weights, constraints)
        if neighbor_score > current_score:
            current_solution, current_score = neighbor, neighbor_score
            steps_without_improvement = 0
        else:
            steps_without_improvement += 1
    return current_solution, current_score

def hill_climbing(solution):
    current_cost = cost(solution)

    best_solution = solution
    best_cost = current_cost

    p = 0.2
    for i in range(MAXITER):
        new_solution = tweak(solution)
        new_cost = cost(new_solution)
        if new_cost >= current_cost and validate(new_solution):
            solution = new_solution
            current_cost = new_cost
            if current_cost > best_cost:
                print(f"Iteration {i}: current cost = {best_cost}, new cost = {current_cost}")
                best_cost = current_cost
                best_solution = solution
        elif validate(new_solution) and np.random.random() < p:
            p = p * 0.99
            solution = new_solution
            current_cost = new_cost
            
    return best_solution

In [309]:
def create_random_valid_solution(num_knapsacks, num_items, weights, constraints):
    solution = np.zeros((num_knapsacks, num_items), dtype=bool)
    shuffled_items = list(range(num_items))
    rng.shuffle(shuffled_items)

    for item_idx in shuffled_items:
        knapsack_idx = rng.integers(0, num_knapsacks)
        solution[knapsack_idx, item_idx] = True
        if not is_valid(solution, num_knapsacks, weights, constraints):
            solution[knapsack_idx, item_idx] = False

    return solution

def void_solution():
    solution = np.zeros((NUM_KNAPSACKS, NUM_ITEMS), dtype=np.bool)
    return solution

In [310]:
def crossover(parent1: np.ndarray, parent2: np.ndarray, num_items) -> np.ndarray:
    child = np.zeros_like(parent1)
    for i in range(num_items):
        if rng.random() < 0.5:
            child[:, i] = parent1[:, i]
        else:
            child[:, i] = parent2[:, i]
    return child

In [311]:
def simulated_annealing_fast(initial_solution: np.ndarray, max_steps: int, initial_temp: float, cooling_rate: float, values, num_knapsacks, weights, constraints, num_items):
    """
    fast version of SA to be used as local search engine.
    """
    current_solution = initial_solution
    current_score = evaluate(current_solution, values, num_knapsacks, weights, constraints)
    best_solution, best_score = current_solution, current_score
    temperature = initial_temp

    for _ in range(max_steps):
        neighbor = move(current_solution, num_items, num_knapsacks)
        neighbor_score = evaluate(neighbor, values, num_knapsacks, weights, constraints)

        #accept always if better, else with a probability
        if neighbor_score > current_score or rng.random() < np.exp((neighbor_score - current_score) / temperature):
            current_solution, current_score = neighbor, neighbor_score

        # update best solution found
        if current_score > best_score:
            best_solution, best_score = current_solution, current_score

        temperature *= cooling_rate
        if temperature < 1e-3:  # avoid too low temperatures
            break

    return best_solution, best_score

In [312]:
# combines the principles of evolution (recombination and selection) with those of individual learning (local search). 
# It balances global exploration (genetic diversity) with local  intensification (solution refinement)
def my_algorithm(generations: int, mu: int, lambda_: int, num_knapsacks, num_items, weights, constraints, values):
    # INITIALIZATION:
    #    - A population of mu feasible random solutions is generated, where each solution encodes an assignment of items to knapsacks
    population = [create_random_valid_solution(num_knapsacks, num_items, weights, constraints) for _ in range(mu)]
    # SOL FOR PROB3: population = [np.zeros((NUM_KNAPSACKS, NUM_ITEMS), dtype=bool) for _ in range(mu)]
    best_solution_so_far, best_score_so_far = None, -1
    # EVOLUTIONARY LOOP 
    for gen in range(generations):
        offspring = []
        for _ in range(lambda_):
            # offspring are created by randomly selecting parent pairs.
            parent1 = population[rng.integers(0, mu)]
            parent2 = population[rng.integers(0, mu)]
            # The crossover operator mixes the parents’ item assignments to produce new offspring inheriting traits from both parents.
            child = crossover(parent1, parent2, num_items)
            # A mutation operator makes small random changes to the offspring solutions.
            child = move(child, num_items, num_knapsacks)
            #optimize child with SA
            # LOCAL SEARCH (memetic phase):
            #- Each offspring is refined using simulated annealing that performs small modifications and occasionally accepts worse solutions to escape local optima. (This allows each individual to "learn" or improve on its own.)
            improved_child, _ = simulated_annealing_fast(child, max_steps=75, initial_temp=10.0, cooling_rate=0.99, values=values, num_knapsacks=num_knapsacks, weights=weights, constraints=constraints, num_items=num_items)
            offspring.append(improved_child)
            # SELECTION:
            #- Parents and offspring are merged into a (mu + lambda) population.
            #- All individuals are evaluated via the objective function.
            # - The mu best solutions are kept for the next generation, ensuring selective pressure toward higher-quality individuals.
        combined_population = population + offspring
        scores = [evaluate(ind, values, num_knapsacks, weights, constraints) for ind in combined_population]
        sorted_indices = np.argsort(scores)[::-1]
        population = [combined_population[i] for i in sorted_indices[:mu]]

        current_best_score = scores[sorted_indices[0]]
        if current_best_score > best_score_so_far:
            best_score_so_far = current_best_score
            best_solution_so_far = population[0]
            print(f"  GENERATION {gen + 1}: new record! value = {best_score_so_far}")

    return best_solution_so_far, best_score_so_far

In [313]:
#PROBLEM1
GENERATIONS = 50
POPULATION_SIZE = 20  # mu (dim of population)
OFFSPRING_SIZE = 200  # lambda (num of children per generation)

best_solution, best_score = my_algorithm(
    generations=GENERATIONS,
    mu=POPULATION_SIZE,
    lambda_=OFFSPRING_SIZE, 
    num_knapsacks=NUM_KNAPSACKS1,
    num_items=NUM_ITEMS1, 
    weights=WEIGHTS1, 
    constraints=CONSTRAINTS1, 
    values=VALUES1
)

if best_score <= 0:
    print("no solution found")
else:
    print("Value:", best_score)

  GENERATION 1: new record! value = 776.0
  GENERATION 2: new record! value = 789.0
  GENERATION 3: new record! value = 827.0
  GENERATION 4: new record! value = 857.0
  GENERATION 5: new record! value = 880.0
Value: 880.0


In [ ]:
# Problem 1:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 3
NUM_ITEMS = 20
NUM_DIMENSIONS = 2
VALUES = rng.integers(0, 100, size=NUM_ITEMS1)
WEIGHTS = rng.integers(0, 100, size=(NUM_ITEMS1, NUM_DIMENSIONS1))
CONSTRAINTS = rng.integers(
    0, 100 * NUM_ITEMS1 // NUM_KNAPSACKS1, size=(NUM_KNAPSACKS1, NUM_DIMENSIONS1)
)

MAXITER= 10000
hill_climbing(np.zeros((NUM_KNAPSACKS, NUM_ITEMS), dtype=np.bool))

Iteration 1: current cost = 0, new cost = 69
Iteration 3: current cost = 69, new cost = 120
Iteration 6: current cost = 120, new cost = 197
Iteration 7: current cost = 197, new cost = 275
Iteration 10: current cost = 275, new cost = 320
Iteration 13: current cost = 320, new cost = 372
Iteration 14: current cost = 372, new cost = 448
Iteration 15: current cost = 448, new cost = 545
Iteration 29: current cost = 545, new cost = 610
Iteration 30: current cost = 610, new cost = 653
Iteration 34: current cost = 653, new cost = 662
Iteration 71: current cost = 662, new cost = 720
Iteration 76: current cost = 720, new cost = 791
Iteration 84: current cost = 791, new cost = 862
Iteration 89: current cost = 862, new cost = 940
Iteration 104: current cost = 940, new cost = 974
Iteration 134: current cost = 974, new cost = 992
Iteration 156: current cost = 992, new cost = 1057
Iteration 354: current cost = 1057, new cost = 1065


array([[ True, False,  True,  True,  True,  True,  True,  True, False,
        False, False, False, False, False,  True,  True,  True, False,
         True, False],
       [False,  True, False, False, False, False, False, False, False,
        False,  True,  True,  True,  True, False, False, False, False,
        False,  True],
       [False, False, False, False, False, False, False, False,  True,
         True, False, False, False, False, False, False, False,  True,
        False, False]])

In [314]:
#PROBLEM2
GENERATIONS = 50
POPULATION_SIZE = 20  # mu (dim of population)
OFFSPRING_SIZE = 200  # lambda (num of children per generation)

best_solution, best_score = my_algorithm(
    generations=GENERATIONS,
    mu=POPULATION_SIZE,
    lambda_=OFFSPRING_SIZE, 
    num_knapsacks=NUM_KNAPSACKS2,
    num_items=NUM_ITEMS2, 
    weights=WEIGHTS2, 
    constraints=CONSTRAINTS2, 
    values=VALUES2
)

if best_score <= 0:
    print("no solution found")
else:
    print("Value:", best_score)

  GENERATION 1: new record! value = 20373.0
  GENERATION 22: new record! value = 21060.0
  GENERATION 23: new record! value = 21617.0
  GENERATION 25: new record! value = 22421.0
  GENERATION 32: new record! value = 23262.0
  GENERATION 34: new record! value = 23601.0
  GENERATION 38: new record! value = 24002.0
  GENERATION 39: new record! value = 24155.0
  GENERATION 41: new record! value = 24549.0
  GENERATION 48: new record! value = 24677.0
  GENERATION 49: new record! value = 24956.0
Value: 24956.0


In [ ]:
# Problem 2:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 10
NUM_ITEMS = 100
NUM_DIMENSIONS = 10
VALUES = rng.integers(0, 1000, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 1000, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    1000 * 2, 1000 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

MAXITER = 100000
solution = void_solution()
solution = hill_climbing(solution)

Iteration 0: current cost = 0, new cost = 839
Iteration 1: current cost = 839, new cost = 1525
Iteration 2: current cost = 1525, new cost = 1837
Iteration 5: current cost = 1837, new cost = 2742
Iteration 6: current cost = 2742, new cost = 3192
Iteration 8: current cost = 3192, new cost = 4118
Iteration 9: current cost = 4118, new cost = 4563
Iteration 10: current cost = 4563, new cost = 5321
Iteration 11: current cost = 5321, new cost = 6102
Iteration 13: current cost = 6102, new cost = 6535
Iteration 16: current cost = 6535, new cost = 7178
Iteration 20: current cost = 7178, new cost = 7372
Iteration 22: current cost = 7372, new cost = 8167
Iteration 24: current cost = 8167, new cost = 8537
Iteration 25: current cost = 8537, new cost = 9395
Iteration 26: current cost = 9395, new cost = 10227
Iteration 27: current cost = 10227, new cost = 10944
Iteration 28: current cost = 10944, new cost = 11020
Iteration 30: current cost = 11020, new cost = 11159
Iteration 34: current cost = 11159, 

In [315]:
#PROBLEM3
initial_solution = create_random_valid_solution(NUM_KNAPSACKS3, NUM_ITEMS3, WEIGHTS3, CONSTRAINTS3)
best_solution, best_score = simulated_annealing_fast(
    initial_solution=initial_solution,
    max_steps=1000,
    initial_temp=100.0,
    cooling_rate=0.995,
    values=VALUES3,
    num_knapsacks=NUM_KNAPSACKS3,
    weights=WEIGHTS3,
    constraints=CONSTRAINTS3,
    num_items=NUM_ITEMS3
)
if best_score <= 0:
    print("no solution found")
else:
    print("Value:", best_score)

Value: 814818.0


In [ ]:
# Problem 3:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 100
NUM_ITEMS = 5000
NUM_DIMENSIONS = 100
VALUES = rng.integers(0, 1000, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 1000, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    1000 * 10, 1000 * 2 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

MAXITER = 20000
solution = void_solution()
solution = hill_climbing(solution)

Iteration 0: current cost = 0, new cost = 648
Iteration 1: current cost = 648, new cost = 1577
Iteration 2: current cost = 1577, new cost = 1777
Iteration 3: current cost = 1777, new cost = 2497
Iteration 4: current cost = 2497, new cost = 3064
Iteration 6: current cost = 3064, new cost = 3168
Iteration 7: current cost = 3168, new cost = 4162
Iteration 12: current cost = 4162, new cost = 5159
Iteration 14: current cost = 5159, new cost = 5163
Iteration 15: current cost = 5163, new cost = 5387
Iteration 18: current cost = 5387, new cost = 5846
Iteration 20: current cost = 5846, new cost = 6795
Iteration 23: current cost = 6795, new cost = 6855
Iteration 24: current cost = 6855, new cost = 7839
Iteration 25: current cost = 7839, new cost = 8342
Iteration 28: current cost = 8342, new cost = 8476
Iteration 29: current cost = 8476, new cost = 8910
Iteration 31: current cost = 8910, new cost = 9020
Iteration 32: current cost = 9020, new cost = 9430
Iteration 33: current cost = 9430, new cost

KeyboardInterrupt: 

In [316]:
"""
OBSERVATIONS: 
Regarding problems 1 and 2, the developed solution is definitely much better than a standard simulated annealing, and in terms of performance/cost, it is worth using it.
However, for the third problem, generating a valid solution is computationally expensive (about 20 seconds per solution), so the algorithm—which needs to generate many of them—becomes unusable in terms of time.
Therefore, we tried two approaches: starting from an invalid solution with our algorithm, and using a standard simulated annealing while trying to optimize it starting from a random but valid initial solution.
When evaluating in terms of cost and performance, the second approach makes more sense.
"""


'\nOBSERVATIONS: \nRegarding problems 1 and 2, the developed solution is definitely much better than a standard simulated annealing, and in terms of performance/cost, it is worth using it.\nHowever, for the third problem, generating a valid solution is computationally expensive (about 20 seconds per solution), so the algorithm—which needs to generate many of them—becomes unusable in terms of time.\nTherefore, we tried two approaches: starting from an invalid solution with our algorithm, and using a standard simulated annealing while trying to optimize it starting from a random but valid initial solution.\nWhen evaluating in terms of cost and performance, the second approach makes more sense.\n'